# Silver Layer — Diagnosis-Related Groups (`drgcodes`)

**Source -> Target:** `<source_schema>.drgcodes` -> `<target_schema>.drgcodes` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Objective:** Standardize DRG assignments while preserving patient identifiers, hospital admissions, DRG systems, codes, descriptions and severity/mortality information.

**Grain:** One source DRG assignment record. Multiple DRG records per admission are permitted; no uniqueness constraint is assumed without validating the data.

**Silver rules:** Preserve all records and clinical meaning, standardize data types and whitespace, validate data quality, and reconcile Bronze–Silver counts. No automatic deletion, deduplication, imputation, filtering or aggregation.

**Duplicate handling:** Exact duplicate rows (all columns identical) are counted and reported in one aggregation pass, but they never block publication and are never removed. The official documentation defines no unique key for this table, so identical rows may be legitimate repeated source records. No row-level duplicate flag is added: duplicates are not defined by the documentation and no profiling evidence supports a flag definition. Profile the reported counts and revisit only if evidence shows a specific, documented duplicate pattern.

**Validation policy:** Row preservation, the ordered schema and the Bronze–Silver agreement are asserted unconditionally. The official `drgcodes` documentation does not declare any column NOT NULL, so NULLs and blank values are reported for review but never block publication.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "drgcodes"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "subject_id": "bigint",
    "hadm_id": "bigint",
    "drg_type": "string",
    "drg_code": "string",
    "description": "string",
    "drg_severity": "int",
    "drg_mortality": "int",
}

In [0]:
# Load and inspect DRG assignment records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

print("Bronze rows:", bronze_df.count())

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- drg_type: string (nullable = true)
 |-- drg_code: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- drg_severity: integer (nullable = true)
 |-- drg_mortality: integer (nullable = true)



subject_id,hadm_id,drg_type,drg_code,description,drg_severity,drg_mortality
10000032,22595853,APR,283,OTHER DISORDERS OF THE LIVER,2,2
10000032,22595853,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,22841357,APR,279,HEPATIC COMA AND OTHER MAJOR ACUTE LIVER DISORDERS,3,2
10000032,22841357,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,25742920,APR,283,OTHER DISORDERS OF THE LIVER,3,2
10000032,25742920,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,29079034,APR,207,OTHER CIRCULATORY SYSTEM DIAGNOSES,3,3
10000032,29079034,HCFA,312,SYNCOPE & COLLAPSE,null,null
10000084,23052089,APR,42,DEGENERATIVE NERVOUS SYSTEM DISORDERS EXCEPT MULTIPLE SCLEROSIS,1,1
10000084,23052089,HCFA,57,DEGENERATIVE NERVOUS SYSTEM DISORDERS W/O MCC,null,null


Bronze rows: 761856


In [0]:
# Profile row counts and missing values
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in bronze_df.columns
    ]
)

display(bronze_profile)

total_rows,null_subject_id,null_hadm_id,null_drg_type,null_drg_code,null_description,null_drg_severity,null_drg_mortality
761856,0,0,0,0,0,394925,394925


In [0]:
# Standardize DRG identifiers, classifications and descriptions
silver_df = bronze_df.select(
    F.col("subject_id").cast("long").alias("subject_id"),
    F.col("hadm_id").cast("long").alias("hadm_id"),
    F.trim(F.col("drg_type").cast("string")).alias("drg_type"),
    F.trim(F.col("drg_code").cast("string")).alias("drg_code"),
    F.trim(F.col("description").cast("string")).alias("description"),
    F.col("drg_severity").cast("int").alias("drg_severity"),
    F.col("drg_mortality").cast("int").alias("drg_mortality")
)

silver_df.printSchema()
display(silver_df.limit(20))

root
 |-- subject_id: long (nullable = true)
 |-- hadm_id: long (nullable = true)
 |-- drg_type: string (nullable = true)
 |-- drg_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- drg_severity: integer (nullable = true)
 |-- drg_mortality: integer (nullable = true)



subject_id,hadm_id,drg_type,drg_code,description,drg_severity,drg_mortality
10000032,22595853,APR,283,OTHER DISORDERS OF THE LIVER,2,2
10000032,22595853,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,22841357,APR,279,HEPATIC COMA AND OTHER MAJOR ACUTE LIVER DISORDERS,3,2
10000032,22841357,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,25742920,APR,283,OTHER DISORDERS OF THE LIVER,3,2
10000032,25742920,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,29079034,APR,207,OTHER CIRCULATORY SYSTEM DIAGNOSES,3,3
10000032,29079034,HCFA,312,SYNCOPE & COLLAPSE,null,null
10000084,23052089,APR,42,DEGENERATIVE NERVOUS SYSTEM DISORDERS EXCEPT MULTIPLE SCLEROSIS,1,1
10000084,23052089,HCFA,57,DEGENERATIVE NERVOUS SYSTEM DISORDERS W/O MCC,null,null


In [0]:
# Report exact duplicate DRG records without removing them or blocking publication
# One aggregation pass over a single groupBy (no exceptAll); duplicates are preserved.
duplicate_summary = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("surplus_rows"),
    )
    .first()
)

duplicate_count = duplicate_summary["duplicate_groups"]
duplicate_surplus_rows = duplicate_summary["surplus_rows"]

print("Exact duplicate DRG groups [preserved, report only]:", duplicate_count)
print("Rows beyond the first in those groups [preserved, report only]:", duplicate_surplus_rows)

Exact duplicate DRG groups [preserved, report only]: 0
Rows beyond the first in those groups [preserved, report only]: 0


In [0]:
# Check missing identifiers, DRG codes and optional clinical fields
quality_df = silver_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in silver_df.columns
    ],
    F.sum(
        F.when(F.col("drg_type") == "", 1).otherwise(0)
    ).alias("blank_drg_types"),
    F.sum(
        F.when(F.col("drg_code") == "", 1).otherwise(0)
    ).alias("blank_drg_codes"),
    F.sum(
        F.when(F.col("description") == "", 1).otherwise(0)
    ).alias("blank_descriptions")
)

display(quality_df)

null_subject_id,null_hadm_id,null_drg_type,null_drg_code,null_description,null_drg_severity,null_drg_mortality,blank_drg_types,blank_drg_codes,blank_descriptions
0,0,0,0,0,394925,394925,0,0,0


In [0]:
# Inspect DRG system distribution and severity/mortality availability
drg_profile = (
    silver_df
    .groupBy("drg_type")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct("hadm_id").alias("admissions"),
        F.sum(F.col("drg_severity").isNull().cast("int")).alias("null_severity"),
        F.sum(F.col("drg_mortality").isNull().cast("int")).alias("null_mortality")
    )
    .orderBy("drg_type")
)

display(drg_profile)

drg_type,records,admissions,null_severity,null_mortality
APR,366931,366931,0,0
HCFA,394925,394925,394925,394925


In [0]:
# Confirm every Bronze row is represented in Silver
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

Bronze rows: 761856
Silver rows: 761856


In [0]:
# Report missing/blank fields (review only) and validate row-count preservation
quality = quality_df.first().asDict()

review_checks = [
    "null_subject_id",
    "null_hadm_id",
    "null_drg_type",
    "null_drg_code",
    "blank_drg_types",
    "blank_drg_codes"
]

review_findings = {k: quality[k] for k in review_checks if quality[k] != 0}
print("NULL/blank findings [preserved, review only; no NOT NULL is documented]:", review_findings or "none")

assert bronze_count == silver_count, "Record counts do not match"

print("Row-preservation check passed.")
print("Exact duplicate groups (preserved, review only):", duplicate_count)

NULL/blank findings [preserved, review only; no NOT NULL is documented]: none
Row-preservation check passed.
Exact duplicate groups (preserved, review only): 0


DataFrame[]

In [0]:
# Publish the validated Silver DRG table (CREATE OR REPLACE; downstream Gold depends on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

Published: cdr_mimiciv.silver.drgcodes


In [0]:
# Verify published DRG records and table structure
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

# No NOT NULL columns or uniqueness key are documented for drgcodes; report published NULLs only
published_quality = published_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in EXPECTED_SCHEMA
    ]
).first()
print("Published NULL counts [preserved, review only]:", published_quality.asDict())

display(published_df.limit(20))

print("Silver drgcodes table verified successfully.")

Bronze rows: 761856
Silver rows: 761856
Published rows: 761856
Published NULL counts [preserved, review only]: {'null_subject_id': 0, 'null_hadm_id': 0, 'null_drg_type': 0, 'null_drg_code': 0, 'null_description': 0, 'null_drg_severity': 394925, 'null_drg_mortality': 394925}


subject_id,hadm_id,drg_type,drg_code,description,drg_severity,drg_mortality
10000032,22595853,APR,283,OTHER DISORDERS OF THE LIVER,2,2
10000032,22595853,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,22841357,APR,279,HEPATIC COMA AND OTHER MAJOR ACUTE LIVER DISORDERS,3,2
10000032,22841357,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,25742920,APR,283,OTHER DISORDERS OF THE LIVER,3,2
10000032,25742920,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA W CC",null,null
10000032,29079034,APR,207,OTHER CIRCULATORY SYSTEM DIAGNOSES,3,3
10000032,29079034,HCFA,312,SYNCOPE & COLLAPSE,null,null
10000084,23052089,APR,42,DEGENERATIVE NERVOUS SYSTEM DISORDERS EXCEPT MULTIPLE SCLEROSIS,1,1
10000084,23052089,HCFA,57,DEGENERATIVE NERVOUS SYSTEM DISORDERS W/O MCC,null,null


Silver drgcodes table verified successfully.


In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")